In [6]:
"This notebook is for selection genes that show bimodal distributions in the corresponding processes."

'This notebook is for selection genes that show bimodal distributions in the corresponding processes.'

In [ ]:
# import scvelo as scv
# import pandas as pd 
# import numpy as np
# from anndata import AnnData
# import loompy
# from matplotlib import pyplot as plt
# from sklearn.preprocessing import StandardScaler,MinMaxScaler
# import matplotlib.patches as mpatches
# import pickle
# import os
# import leidenalg
# from scipy import sparse
# from IPython.display import SVG
# from sklearn.linear_model import LinearRegression
# from scipy import stats,signal
# from sklearn.preprocessing import StandardScaler,MinMaxScaler
# from sklearn.cross_decomposition import PLSRegression
# from sklearn.feature_selection import f_regression, mutual_info_regression
# from sklearn.cluster import KMeans

# from sklearn.mixture import GaussianMixture
# from sklearn.metrics import silhouette_score
# from filter_dispersion import filter_dispersion
# from scipy.sparse import issparse
# from scvelo.preprocessing.utils import get_mean_var,materialize_as_ndarray
# from sklearn.cluster import MeanShift
# from scipy.stats import gaussian_kde, norm
# from scipy.signal import find_peaks, peak_prominences

In [46]:
import os
import anndata as ad
import matplotlib.pyplot as plt
from filter_dispersion import filter_dispersion
import numpy as np
import scvelo as scv
from scipy.sparse import issparse
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [47]:
if not os.path.exists('result/'):
    os.makedirs('result/')
result_path='result/'

In [48]:
adata0=scv.read_loom("Data/Loom_data/a549_tgfb1.loom")

In [49]:
print(adata0)
print("Obs : ", adata0.obs.shape)
print("Var : ", adata0.var.shape)
print("Layers : ", list(adata0.layers.keys()))

AnnData object with n_obs × n_vars = 3567 × 33694
    obs: 'Clusters', '_X', '_Y', 'batch', 'obs_names', 'treatment'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'var_names'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced'
Obs :  (3567, 6)
Var :  (33694, 6)
Layers :  ['matrix', 'ambiguous', 'spliced', 'unspliced']


In [50]:
"selecting the high dispersion genes in the CPT"

'selecting the high dispersion genes in the CPT'

In [51]:
adata=adata0.copy()
adata,df,gene_subset=filter_dispersion(adata,n_bins=20,min_disp = 0.5,max_disp = np.inf,min_mean = 0.01,max_mean = 3)

# adata.var.index.values[gene_subset].shape
hv_genes=adata.var.index.values[gene_subset]
adata=adata[:,hv_genes]

#----distribution of shared counts

Xs, Xu = adata.layers["spliced"], adata.layers["unspliced"]
nonzeros = (
    (Xs > 0).multiply(Xu > 0) if issparse(Xs) else (Xs > 0) * (Xu > 0)
            )
X = (
    nonzeros.multiply(Xs) + nonzeros.multiply(Xu)
    if issparse(nonzeros)
                else nonzeros * (Xs + Xu)
            )

# plt.hist(np.sum(X.A,axis=0),bins=100,edgecolor='k')

np.where(np.log(np.sum(X.A,axis=0))>2)[0].shape

(1199,)

In [52]:
print("adata shape:", adata.shape)
print("genes retained:", np.sum(gene_subset))
print(df.head())

adata shape: (3567, 1699)
genes retained: 1699
           mean  dispersion           mean_bin  dispersion_norm
0  1.000000e-12         NaN  (-0.00532, 0.266]         0.000000
1  1.000000e-12         NaN  (-0.00532, 0.266]         0.000000
2  1.000000e-12         NaN  (-0.00532, 0.266]         0.000000
3  1.000000e-12         NaN  (-0.00532, 0.266]         0.000000
4  7.540882e-03   -0.007318  (-0.00532, 0.266]        -0.357678


In [53]:
adata=adata0[:,hv_genes]
scv.pp.filter_and_normalize(adata,min_shared_counts=10)


print(adata.X.shape)
scv.pp.neighbors(adata, n_neighbors=30)
scv.pp.pca(adata,n_comps=50)
scv.pp.moments(adata, n_pcs=50, n_neighbors=30)

# scv.tl.umap(adata)
# scv.tl.recover_dynamics(adata)

scv.tl.velocity(adata,mode='stochastic',perc=[5, 95])

scv.tl.velocity_graph(adata,xkey='Ms')

Filtered out 541 genes that are detected 10 counts (shared).
Normalized count data: X, spliced, unspliced.
Logarithmized X.
(3567, 1158)
computing neighbors
    finished (0:00:00) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
computing velocities
    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
computing velocity graph (using 1/32 cores)


  0%|          | 0/3567 [00:00<?, ?cells/s]

    finished (0:00:01) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)


In [54]:
print(adata.layers.keys())
print(adata.uns.keys())
print(adata)

KeysView(Layers with keys: matrix, ambiguous, spliced, unspliced, Ms, Mu, velocity, variance_velocity)
odict_keys(['pca', 'neighbors', 'velocity_params', 'velocity_graph', 'velocity_graph_neg'])
AnnData object with n_obs × n_vars = 3567 × 1158
    obs: 'Clusters', '_X', '_Y', 'batch', 'obs_names', 'treatment', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts', 'velocity_self_transition'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'var_names', 'velocity_gamma', 'velocity_qreg_ratio', 'velocity_r2', 'velocity_genes'
    uns: 'pca', 'neighbors', 'velocity_params', 'velocity_graph', 'velocity_graph_neg'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced', 'Ms', 'Mu', 'velocity', 'variance_velocity'
    obsp: 'distances', 'connectivities'


In [55]:
"selecting genes that can be binarized"

'selecting genes that can be binarized'

In [56]:
gene_arr=adata.var.index.values

In [57]:
from critical_bandwidth import critical_bandwidth,critical_bandwidth_m_modes

from sklearn.neighbors import KernelDensity
from scipy.signal import argrelextrema
def best_split(data, I=(-np.inf, np.inf)):
    '''With bimodal data, finding split at lowest density.'''
    h_crit = critical_bandwidth_m_modes(data, 2, I)
    kde = KernelDensity(kernel='gaussian', bandwidth=h_crit).fit(data.reshape(-1, 1))
    x = np.linspace(max(np.min(data), I[0]), min(np.max(data), I[1]), 200)
    y = np.exp(kde.score_samples(x.reshape(-1, 1)))
    modes = argrelextrema(np.hstack([[0], y, [0]]), np.greater)[0]
    ind_min=[]
    if len(modes)>=2: 
        for i in range(len(modes)-1):
            if len(modes[i]-1 + argrelextrema(y[(modes[i]-1):(modes[i+1]-1)], np.less)[0])>0:
                ind_min.append(modes[i]-1 + argrelextrema(y[(modes[i]-1):(modes[i+1]-1)], np.less)[0][0])
#     else:
#         if len(modes[0]-1 + argrelextrema(y[(modes[0]-1):(modes[1]-1)], np.less)[0])>0:
#             ind_min = modes[0]-1 + argrelextrema(y[(modes[0]-1):(modes[1]-1)], np.less)[0]
#     print(ind_min)
    return x[ind_min]

In [58]:
X0_ori=adata.layers['Ms']
X0_bin=X0_ori.copy()
EG_bin_genes_center=[]
EG_bin_genes=[]
for i in range(X0_ori.shape[1]):
    if i%500==0:
        print(i)

    
    mask=X0_ori[:,i]>0
    # plt.hist(X0_ori[:,i][mask],bins=50,edgecolor='k')
    # plt.show()
    
    x=X0_ori[:,i]
    split_x=best_split(x)

    kmeans = KMeans(n_clusters=2, random_state=0).fit(X0_ori[:,i][:,None])
    klabels=kmeans.labels_
    kcenters = np.sort(np.array([i[0] for i in kmeans.cluster_centers_]))
    cl_score=silhouette_score(X0_ori[:,i][:,None], klabels, metric="euclidean")

    bin_flag=0        
    if len(split_x)>0 and cl_score>0.65:
        for si in range(len(split_x)):
            if kcenters[0]<split_x[si] and kcenters[1]>split_x[si]:
                bin_flag=1

                break
    if bin_flag==1:
        
        EG_bin_genes.append(gene_arr[i])

        EG_bin_genes_center.append([kcenters[0],kcenters[1]])

            
        

0
500
1000


In [59]:
EG_bin_genes_center=np.array(EG_bin_genes_center)
EG_bin_genes=np.array(EG_bin_genes)
np.save(result_path+'EG_bin_genes.npy',EG_bin_genes)
np.save(result_path+'EG_bin_genes_center.npy',EG_bin_genes_center)

In [60]:
print(len(EG_bin_genes),len(np.unique(EG_bin_genes)))

41 41


In [63]:
adata=adata0[:,EG_bin_genes]

scv.pp.filter_and_normalize(adata,min_shared_counts=10)#min_shared_cells=150,,n_top_genes=1000#, 
print(adata.X.shape)
scv.pp.neighbors(adata, n_neighbors=20)
scv.pp.pca(adata,n_comps=40)
scv.pp.moments(adata, n_pcs=50, n_neighbors=20)
# scv.tl.umap(adata)
scv.tl.velocity(adata,mode='stochastic',perc=[5, 95])

scv.tl.velocity_graph(adata,xkey='Ms')

Normalized count data: X, spliced, unspliced.
Logarithmized X.
(3567, 41)
computing neighbors
    finished (0:00:00) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
computing velocities
    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
computing velocity graph (using 1/32 cores)


  0%|          | 0/3567 [00:00<?, ?cells/s]

    finished (0:00:00) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)


In [65]:
scv.tl.velocity_pseudotime(adata)
# scv.pl.scatter(adata, basis='umap',color='velocity_pseudotime', color_map='gnuplot')
# scv.tl.latent_time(adata)
# scv.pl.scatter(adata, basis='pca',color='latent_time', color_map='jet', size=80)
# scv.pl.scatter(adata, basis='umap',color='clusters')
# scv.pl.scatter(adata, basis='pca',color='clusters')
# scv.pl.velocity_embedding(adata, basis='pca',color='clusters')

# scv.pl.velocity_embedding_stream(adata, basis='pca',color='clusters',s=50,layer='spliced',color_map='jet')

In [67]:
adata.obsm['X_pca'][:,0]=-adata.obsm['X_pca'][:,0]
# scv.pl.velocity_graph(adata, basis='pca',color='clusters',dpi=300,save='EG_vgraph.png')

In [68]:
adata.write_h5ad(result_path+'EG_bin.h5ad')#h5ad save all features

In [69]:
adata0_sel=adata0[:,EG_bin_genes]
adata0_sel.write_h5ad(result_path+'EG_ori_bin.h5ad')
adata0_sel

AnnData object with n_obs × n_vars = 3567 × 41
    obs: 'Clusters', '_X', '_Y', 'batch', 'obs_names', 'treatment'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'var_names'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced'

In [73]:
import anndata as ad

adata = ad.read_h5ad("result/EG_ori_bin.h5ad")
print(adata)
print(adata.layers.keys())

AnnData object with n_obs × n_vars = 3567 × 41
    obs: 'Clusters', '_X', '_Y', 'batch', 'obs_names', 'treatment'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'var_names'
    layers: 'ambiguous', 'matrix', 'spliced', 'unspliced'
KeysView(Layers with keys: ambiguous, matrix, spliced, unspliced)


In [74]:
adata = ad.read_h5ad("result/EG_ori_bin.h5ad")

print(adata.shape)
print(adata.var_names[:10])

(3567, 41)
Index(['295', '701', '2185', '3695', '6680', '8077', '8407', '8459', '8736',
       '8922'],
      dtype='object')
